In [1]:
import heapq #for retrieving the smallest priority efficiently
from collections import deque
import time
# Maps for the test
MAP_A = [
    "S..#....",
    "##.#.##.",
    "....~~..",
    ".####.#.",
    "...~..#G"
]
MAP_B = [
    "S~~~~~~~G",
    ".#######.",
    ".........",
    ".#.#.#.#.",
    "........."
]
MAP_C = [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G"
]

In [2]:
# To change the map: set GRID to MAP_A / MAP_B / MAP_C 
GRID = MAP_A
RUN_ALL_MAPS = True  # True: run on all 3 maps | False: run only on GRID

In [3]:
# Helper Functions 
def find_start_goal(grid):
    start = goal = None
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell == "S":
                start = (r, c)
            if cell == "G":
                goal = (r, c)
    return start, goal
# Calculate the cost of entering a cell
def get_cost(grid, position):
    r, c = position
    if grid[r][c] == '~':
        return 3
    return 1

In [4]:
#check whether a position is valid
def is_valid(r, c, grid):
    return 0 <= r < len(grid) and 0 <= c < len(grid[0]) and grid[r][c] != '#'

In [5]:
# Calculate the Manhattan distance to the goal
def heuristic(position,goal):
    r,c=position
    r_g,c_g=goal
    distance=abs(r_g-r)+abs(c_g-c)
    return distance

In [6]:
# find neighboring postions 
def get_neighbors(grid,position):
    r,c=position
    
    directions=[ (-1,0) ,#up
                 (1,0),#down
                 (0,1),#right
                 (0,-1)#left
     ]
    neighbors=[] #valid neighboring positions into it
    for dr,dc in directions:
        new_r=r+dr
        new_c=c+dc
        if is_valid(new_r,new_c,grid):
            neighbors.append((new_r,new_c))
    return neighbors        

In [7]:
def A_star(grid):
    start, goal = find_start_goal(grid)

    if start == None or goal == None:
        return None, None, 0 ,0

    frontier = []  # positions waiting to be explored
    heapq.heappush(frontier, (heuristic(start, goal), start))

    came_from = {}  # Stores the previous position for each position
    g_cost = {start: 0}  # Stores the cheapest known cost to reach each position
    expanded_nodes = 0
    max_frontier=1 #the frontier contains only the starting position

    while frontier:
        current_priority, current = heapq.heappop(frontier)

        if current_priority != g_cost[current] + heuristic(current, goal):
            continue

        expanded_nodes += 1

        if current == goal:
            break

        for neighbor in get_neighbors(grid, current):
            new_cost = g_cost[current] + get_cost(grid, neighbor)  # new cost = current path cost + cost of entering the neighbor

            if neighbor not in g_cost or new_cost < g_cost[neighbor]:
                g_cost[neighbor] = new_cost
                came_from[neighbor] = current
                priority = new_cost + heuristic(neighbor, goal)
                heapq.heappush(frontier, (priority, neighbor))
                max_frontier=max(max_frontier,len(frontier))

    if goal not in came_from and start != goal:  # goal was never reached (no path was found)
        return None, None, expanded_nodes,max_frontier

    path = []
    current = goal  # backward from the goal to the start

    while current != start:
        path.append(current)
        current = came_from[current]

    path.append(start)
    path.reverse()

    return path, g_cost[goal], expanded_nodes,max_frontier
def BFS(grid):
    start, goal = find_start_goal(grid)
    if start is None or goal is None:
        return None, None, 0, 0

    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right

    queue = deque([(start, [start])])
    visited = set([start])
    nodes_expanded = 0
    max_frontier = 1  # the queue contains only the starting position

    while queue:
        (r, c), path = queue.popleft()
        nodes_expanded += 1

        if (r, c) == goal:
            total_cost = 0
            for i in range(1, len(path)):
                total_cost += get_cost(grid, path[i])
            return path, total_cost, nodes_expanded, max_frontier

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if is_valid(nr, nc, grid) and (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [(nr, nc)]))
                max_frontier = max(max_frontier, len(queue))

    return None, None, nodes_expanded, max_frontier

In [8]:
#  Display Path 
def display_path(grid, path):
    if path is None:
        print("No path found")
        return

    display_grid = [list(row) for row in grid]
    for r, c in path:
        if display_grid[r][c] not in ('S', 'G'):
            display_grid[r][c] = '*'

    print("Path on map:")
    for row in display_grid:
        print(''.join(row))

In [9]:
#  Run 
if RUN_ALL_MAPS:
    maps = {
        "MAP_A": MAP_A,
        "MAP_B": MAP_B,
        "MAP_C": MAP_C
    }
else:
    maps = {"GRID": GRID}

In [10]:
algorithms = {
    "BFS": BFS,
    "A*": A_star
}

In [11]:
results = []  # rows of the results table

In [14]:
for map_name, grid in maps.items():
    for algo_name, algo in algorithms.items():
        print("\n" + "="*50)
        print("Algorithm:", algo_name)
        print("Map:", map_name)
        print("="*50)

        start_time = time.perf_counter()
        path, total_cost, expanded_nodes, max_frontier = algo(grid)
        end_time = time.perf_counter()
        runtime = end_time - start_time
        if path is None:
            
            print("Path: No path found")
            print("Path length: N/A")
            print("Total cost: N/A")
            print("Expanded nodes:", expanded_nodes)
            print("Max frontier size:", max_frontier)
            print(f"Runtime: {runtime:.6f} seconds")

            results.append((map_name, algo_name, "N/A", "N/A",
            expanded_nodes, max_frontier, runtime) )
   
 
        else:
            print("Path:", path)
            print("Path length:", len(path) - 1)
            print("Cost:", total_cost)
            print("Expanded nodes:", expanded_nodes)
            print("Max frontier size:", max_frontier)
            print(f"Runtime: {runtime:.6f} seconds")
            display_path(grid, path)
            results.append((map_name, algo_name, len(path) - 1, total_cost, expanded_nodes, max_frontier, runtime))


Algorithm: BFS
Map: MAP_A
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length: 11
Cost: 15
Expanded nodes: 28
Max frontier size: 5
Runtime: 0.000150 seconds
Path on map:
S**#....
##*#.##.
..******
.####.#*
...~..#G

Algorithm: A*
Map: MAP_A
Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length: 11
Cost: 15
Expanded nodes: 21
Max frontier size: 5
Runtime: 0.000191 seconds
Path on map:
S**#....
##*#.##.
..******
.####.#*
...~..#G

Algorithm: BFS
Map: MAP_B
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8)]
Path length: 8
Cost: 22
Expanded nodes: 25
Max frontier size: 4
Runtime: 0.000112 seconds
Path on map:
S*******G
.#######.
.........
.#.#.#.#.
.........

Algorithm: A*
Map: MAP_B
Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]
Path length: 12
Cost: 12
Expanded nodes: 15
Max

In [15]:
# Results table
print("\n" + "="*85)
print("RESULTS TABLE")
print("="*85)
print(f"{'Map':<8}{'Algorithm':<11}{'Length':<8}{'Cost':<7}{'Expanded':<10}{'MaxFrontier':<13}{'Runtime(s)':<12}")
print("-"*85)
for m, a, l, c, e, f, t in results:
    print(f"{m:<8}{a:<11}{str(l):<8}{str(c):<7}{e:<10}{f:<13}{t:<12.6f}")


RESULTS TABLE
Map     Algorithm  Length  Cost   Expanded  MaxFrontier  Runtime(s)  
-------------------------------------------------------------------------------------
MAP_A   BFS        11      15     28        5            0.000173    
MAP_A   A*         11      15     21        5            0.000248    
MAP_B   BFS        8       22     25        4            0.000082    
MAP_B   A*         12      12     15        7            0.000081    
MAP_C   BFS        -       -      13        3            0.000045    
MAP_C   A*         -       -      13        3            0.000061    
MAP_A   BFS        11      15     28        5            0.000150    
MAP_A   A*         11      15     21        5            0.000191    
MAP_B   BFS        8       22     25        4            0.000112    
MAP_B   A*         12      12     15        7            0.000126    
MAP_C   BFS        N/A     N/A    13        3            0.000063    
MAP_C   A*         N/A     N/A    13        3            0.